<h1 style="font-family:verdana;"> <center>🛍 OTTO – Multi-Objective Recommender System - Getting Started 🧑‍💻</center> </h1>

***

<div style="color:white;
           display:fill;
           border-radius:5px;
           background-color:#0daae3;
           font-size:110%;
           font-family:Verdana;
           letter-spacing:0.5px">
        <p style="padding: 10px;
              color:white;">
            Hopefully this notebook will give you a basic understanding of the task and data involved in this competition. Please give an upvote if you find it useful 👍
        </p>
    </div>
    
<div align = 'center'><img src= "https://livewin.net/wp-content/uploads/2021/02/Ecommerce.png" alt ="Shop" style='width: 1000px;height 500px'>

### <span style="font-family:verdana; word-spacing:1.5px;"> Contents:
[Load in the data ⏳](#first-bullet)
    
[Data Structure 🗂](#second-bullet)
    
[Intital EDA 📊](#third-bullet)
    
[Baseline 📈](#fourth-bullet)
    
[Where to go next 🚀](#fith-bullet)

### <span style="font-family:verdana; word-spacing:1.5px;">  Task overview
    
<span style="font-family:verdana; word-spacing:1.5px;">  The aim of this competition is to predict e-commerce <span style="color:#159364;">clicks, cart additions, and orders</span>. You'll build a multi-objective recommender system based on previous events in a user session.
    
<span style="font-family:verdana; word-spacing:1.5px;"> Current recommender systems consist of various models with different approaches, ranging from simple matrix factorization to a transformer-type deep neural network. However, no single model exists that can simultaneously optimize multiple objectives. In this competition, you’ll build a single entry to predict click-through, add-to-cart, and conversion rates based on previous same-session events.

### <span style="font-family:verdana; word-spacing:1.5px;">   Imports / setup 🚚

In [ ]:
### Imports ###

import pandas as pd
from pathlib import Path
import os
import random
import numpy as np
import json
from datetime import timedelta
from collections import Counter
from tqdm.notebook import tqdm
from heapq import nlargest

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme()

import warnings
warnings.filterwarnings('ignore')


In [ ]:
### Paths ###

DATA_PATH = Path('../input/otto-recommender-system')
TRAIN_PATH = DATA_PATH/'train.jsonl'
TEST_PATH = DATA_PATH/'test.jsonl'
SAMPLE_SUB_PATH = Path('../input/otto-recommender-system/sample_submission.csv')

# <span style="font-family:verdana; word-spacing:1.5px;">   Load in the data ⏳

In [ ]:
# Lets check how many lines the training data has!

with open(TRAIN_PATH, 'r') as f:
    print(f"We have {len(f.readlines()):,} lines in the training data")

In [ ]:
# Load in a sample to a pandas df

sample_size = 150000

chunks = pd.read_json(TRAIN_PATH, lines=True, chunksize = sample_size)

for c in chunks:
    sample_train_df = c
    break

In [ ]:
sample_train_df.set_index('session', drop=True, inplace=True)
sample_train_df.head()

# <span style="font-family:verdana; word-spacing:1.5px;">   Data structure 🗂
    
<span style="font-family:verdana; word-spacing:1.5px;">  `session` - the unique session id. Each session contains a list of time ordered events.

<span style="font-family:verdana; word-spacing:1.5px;">  `events` - the time ordered sequence of events in the session. Each event contains 3 pieces of information:

- <span style="font-family:verdana; word-spacing:1.5px;">  `aid` - the article id (product code) of the associated event
    
- <span style="font-family:verdana; word-spacing:1.5px;">  `ts` - the Unix timestamp of the event (Unix time is the number of **milliseconds** that have elapsed since 00:00:00 UTC on 1 January 1970) (Thanks to Junji Takeshima https://www.kaggle.com/junjitakeshima for the correction)

- <span style="font-family:verdana; word-spacing:1.5px;">  `type` - the event type, i.e., whether a product was clicked (`clicks`), added to the user's cart (`carts`), or ordered during the session (`orders`)


In [ ]:
# Let's look at an example session and print out some basic info

# Sample the first session in the df
example_session = sample_train_df.iloc[0].item()
print(f'This session was {len(example_session)} actions long \n')
print(f'The first action in the session: \n {example_session[0]} \n')

# Time of session
time_elapsed = example_session[-1]["ts"] - example_session[0]["ts"]
# The timestamp is in milliseconds since 00:00:00 UTC on 1 January 1970
print(f'The first session elapsed: {str(timedelta(milliseconds=time_elapsed))} \n')

# Count the frequency of actions within the session
action_counts = {}
for action in example_session:
    action_counts[action['type']] = action_counts.get(action['type'], 0) + 1  
print(f'The first session contains the following frequency of actions: {action_counts}')

# <span style="font-family:verdana; word-spacing:1.5px;"> Intital EDA 📊

In [ ]:
### Extract information from each session and add it to the df ###

action_counts_list, article_id_counts_list, session_length_time_list, session_length_action_list = ([] for i in range(4))
overall_action_counts = {}
overall_article_id_counts = {}

for i, row in tqdm(sample_train_df.iterrows(), total=len(sample_train_df)):
    
    actions = row['events']
    
    # Get the frequency of actions and article_ids
    action_counts = {}
    article_id_counts = {}
    for action in actions:
        action_counts[action['type']] = action_counts.get(action['type'], 0) + 1
        article_id_counts[action['aid']] = article_id_counts.get(action['aid'], 0) + 1
        overall_action_counts[action['type']] = overall_action_counts.get(action['type'], 0) + 1
        overall_article_id_counts[action['aid']] = overall_article_id_counts.get(action['aid'], 0) + 1
        
    # Get the length of the session
    session_length_time = actions[-1]['ts'] - actions[0]['ts']
    
    # Add to list
    action_counts_list.append(action_counts)
    article_id_counts_list.append(article_id_counts)
    session_length_time_list.append(session_length_time)
    session_length_action_list.append(len(actions))
    
sample_train_df['action_counts'] = action_counts_list
sample_train_df['article_id_counts'] = article_id_counts_list
sample_train_df['session_length_unix'] = session_length_time_list
sample_train_df['session_length_hours'] = sample_train_df['session_length_unix']*2.77778e-7  # Convert to hours
sample_train_df['session_length_action'] = session_length_action_list

In [ ]:
### Actions ###

total_actions = sum(overall_action_counts.values())

plt.figure(figsize=(8,6))
sns.barplot(x=list(overall_action_counts.keys()), y=[i/total_actions for i in overall_action_counts.values()]);
plt.title(f'Action frequency', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xlabel('Category', fontsize=12)
plt.show()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(24, 10))

p = sns.distplot(sample_train_df['session_length_action'], color="y", bins= 70, ax=ax[0], kde=False)
p.set_xlabel("Number of actions", fontsize = 16)
p.set_ylabel("Density", fontsize = 16)
p.set_title("Distribution of the number of actions taken in each session", fontsize = 14)
p.axvline(sample_train_df['session_length_action'].mean(), color='r', linestyle='--', label="Mean")

p = sns.distplot(sample_train_df['session_length_hours'], color="b", bins= 70, ax=ax[1], kde=False)
p.set_xlabel("Hours", fontsize = 16)
p.set_ylabel("Density", fontsize = 16)
p.set_title("Length of each session", fontsize = 16);

Something seems a bit odd with the minutes plot. All the sessions are capped at 650 hours - this needs looking into .. 🤔

In [ ]:
print(f'{round(len(sample_train_df[sample_train_df["session_length_action"]<10])/len(sample_train_df),3)*100}% of the sessions had less than 10 actions')

In [ ]:
article_id_freq = list(overall_article_id_counts.values())
cut_off = [i for i in article_id_freq if i<30]

plt.figure(figsize=(8,6))
sns.distplot(cut_off, bins=30, kde=False);
plt.title(f'Article ID frequency', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xlabel('Article', fontsize=12);

As we can see from the plot above the vast majority of atricles have a very small number of actions relating to them. There are some exceptions..

In [ ]:
### Look at the most interacted with articles ###
print(f'Frequency of most common articles: {sorted(list(overall_article_id_counts.values()))[-5:]} \n')
res = nlargest(5, overall_article_id_counts, key = overall_article_id_counts.get)
print(f'IDs for those common articles: {res}')

# <span style="font-family:verdana; word-spacing:1.5px;">   Baseline 📈
    
<span style="font-family:verdana; word-spacing:1.5px;"> The `test data` contains truncated session data similar to that of the training data. The task is to <span style="color:#159364;"> predict the next aid clicked </span> after the session truncation, as well as the the remaining aids that are added to carts and orders; you may predict up to 20 values for each session type
    
<span style="font-family:verdana; word-spacing:1.5px;"> Submissions are evaluated on <span style="color:#159364;"> Recall </span> each action type, and the three recall values are weight-averaged: <span style="color:#159364;"> {'clicks': 0.10, 'carts': 0.30, 'orders': 0.60} </span>. It is important to get the 'orders' predictions correct as they carry most of the weigthing :)
    
<span style="font-family:verdana; word-spacing:1.5px;"> For each <span style="color:#159364;"> session</span> in the test data, your task it to predict the <span style="color:#159364;"> aid</span> values for each <span style="color:#159364;"> type</span> that occur after the last timestamp ts the test session. In other words, the test data contains sessions truncated by timestamp, and you are to predict what occurs after the point of truncation.

<span style="font-family:verdana; word-spacing:1.5px;"> For <span style="color:#159364;"> clicks </span>there is only a single ground truth value for each session, which is the next <span style="color:#159364;"> aid</span> clicked during the session (although you can still predict up to 20 aid values). The ground truth for <span style="color:#159364;"> carts</span> and <span style="color:#159364;"> orders</span> contains all <span style="color:#159364;"> aid </span>values that were added to a cart and ordered respectively during the session.
    
Each session and type combination should appear on its own session_type row in the submission (3 rows per session), and predictions should be space delimited. This can be seen in the `sample_test_df` below..

In [ ]:
with open(TEST_PATH, 'r') as f:
    print(f"We have {len(f.readlines()):,} lines in the test data")

In [ ]:
# Load in a sample to a pandas df

sample_size = 150

chunks = pd.read_json(TEST_PATH, lines=True, chunksize = sample_size)

for c in chunks:
    sample_test_df = c
    break

In [ ]:
sample_test_df.head()

<span style="font-family:verdana; word-spacing:1.5px;"> Below shows a sample submission. For each `session` in the test set there is a prediction (`labels`). This predicts what articles will be next interacted with in that session. For each session there are three actions (clicks, carts, orders), predictions are made for all three actions.

In [ ]:
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)
sample_submission.head()

<span style="font-family:verdana; word-spacing:1.5px;"> Lets find the most common article for each different type of action.

In [ ]:
sample_size = 150000

chunks = pd.read_json(TRAIN_PATH, lines=True, chunksize = sample_size)

clicks_article_list = []
carts_article_list = []
orders_article_list = []

for e, c in enumerate(chunks):
    
    # Save time by not using all the data
    if e > 20:
        break
    
    sample_train_df = c
    
    for i, row in c.iterrows():
        actions = row['events']
        for action in actions:
            if action['type'] == 'clicks':
                clicks_article_list.append(action['aid'])
            elif action['type'] == 'carts':
                carts_article_list.append(action['aid'])
            else:
                orders_article_list.append(action['aid'])
    

In [ ]:
# Create dictionaries with articles and their frequencies
article_click_freq = Counter(clicks_article_list)
article_carts_freq = Counter(carts_article_list)
article_order_freq = Counter(orders_article_list)

In [ ]:
# Get the 20 most frequent articles for each action
top_click_article = nlargest(20, article_click_freq, key = article_click_freq.get)
top_carts_article = nlargest(20, article_carts_freq, key = article_carts_freq.get)
top_order_article = nlargest(20, article_order_freq, key = article_order_freq.get) 

In [ ]:
# Create a dict with this info
frequent_articles = {'clicks': top_click_article, 'carts':top_carts_article, 'order':top_order_article}

In [ ]:
for action in ['clicks', 'carts', 'order']:
    print(f'Most frequent articles for {action}: {frequent_articles[action][:5]}') # Correction by @danielliao 🙏

<span style="font-family:verdana; word-spacing:1.5px;"> There is some overlap but the articles do change for the different actions!
    
<span style="font-family:verdana; word-spacing:1.5px;"> This baseline will use the fact that people will often interact with articles they have previouslt interacted with. The prediction will consist of the top 20 most frequent articles in the session. If there are less than 20 articles in the session the prediction will be padded with the most frequent articles in the training data as found above.

In [ ]:
test_data = pd.read_json(TEST_PATH, lines=True, chunksize=1000)

preds = []

for chunk in tqdm(test_data, total=1671):
    
    for i, row in chunk.iterrows():
        actions = row['events']
        article_id_list = []
        for action in actions:
            article_id_list.append(action['aid'])
            
        # Get 20 most common article ID for the session
        article_freq = Counter(article_id_list)
        top_articles = nlargest(20, article_freq, key = article_freq.get)
        
        # Pad with most popular items in training
        padding_size = (20 - len(top_articles)) # Correction by @danielliao 🙏
        for action in ['clicks', 'carts', 'order']:
            top_articles_added = top_articles + frequent_articles[action][:padding_size] # Correction by @danielliao 🙏
            preds.append(" ".join([str(id) for id in top_articles_added]))

In [ ]:
# Predict the 20 most common atricles for each test session
sample_submission['labels'] = preds

In [ ]:
sample_submission.to_csv('submission.csv', index=False)

# Where to go next 🚀

- At the moment we pad every prediction with the same most frequently occuring articles. The model could be improved if we looked at which articles co-occured frequently in the same session ([KJ](https://www.kaggle.com/code/whitelily/co-occurrence-baseline) and [VLADIMIR SLAYKOVSKIY](https://www.kaggle.com/code/vslaykovsky/co-visitation-matrix) have started looking at this).

- While shopping there is a common sequence of events: click -> cart -> order. Currently we look at which articles are most common for each action but surely the most likely item to be ordered is one already in the cart 🤔


# WIP :)